## Growth Predictor Model Training

Forecasts the growth rate of an asset by 1 year, and is designed to provide a float value which fits directly into the Intrinsic Value Analyzer model's input data.

Collected data:
- Tickers, and end-of-year share prices from 2019 till 2023
- model trained on 4 CAGR figures, to forecast the 5th CAGR, next years
- model test data (labels) will come from 2024 end-of-year price
- label calculated from 2023 - 2024 price change or CAGR

#### Imports

In [33]:
from os import path
from csv import DictReader

import logging
import pandas as pd
import numpy as np
import yfinance as yf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score
from sklearn.metrics import recall_score, roc_auc_score
from sklearn.metrics import confusion_matrix
from xgboost import XGBClassifier

#### Loading Data from API

In [21]:
data_file = path.join("..", "data")
snp500_path = path.join(data_file, "constituents.csv")
nasdaq_path = path.join(data_file, "nasdaq-listed.csv")
output_path = path.join(data_file, "tickers.csv")
collected_tks = []
existing_tks = set()


with open(snp500_path, mode="r", encoding="utf-8") as f:
    reader = DictReader(f)
    for row in reader:
        ticker = row["Symbol"].strip().upper()
        if ticker and ticker not in existing_tks:
            collected_tks.append(ticker)
            existing_tks.add(ticker)


nasdaq_added_count = 0
with open(nasdaq_path, mode="r", encoding="utf-8") as f:
    reader = DictReader(f)
    for row in reader:
        if nasdaq_added_count >= 500: 
            break
        ticker = row["Symbol"].strip().upper()
        if ticker and ticker not in existing_tks:
            collected_tks.append(ticker)
            existing_tks.add(ticker)
            nasdaq_added_count += 1


print(f"Tickers loaded from {snp500_path} and {nasdaq_path}.")

Tickers loaded from ..\data\constituents.csv and ..\data\nasdaq-listed.csv.


In [22]:
logger = logging.getLogger('yfinance')
logger.setLevel(logging.CRITICAL)
all_price_records = []


try:
    data = yf.download(collected_tks, start="2019-01-09", 
                       end="2025-01-08", group_by='column', progress=False)
    closing = data['Close'] if isinstance(data.columns, pd.MultiIndex) else data
    closing.index = pd.to_datetime(closing.index)
    end_prices = closing.groupby(closing.index.year).last()
    final_df = end_prices.T
    final_df.reset_index(inplace=True)
    final_df.rename(columns={'index': 'Ticker'}, inplace=True)
    final_df.columns = [str(col) for col in final_df.columns]
    columns_order = ['Ticker', '2019', '2020', '2021', '2022', '2023', '2024']
    final_df = final_df.reindex(columns=columns_order)
    final_df.dropna(subset=columns_order, inplace=True)
    final_df.to_csv(output_path, index=False)
    print(f"Saved {len(final_df)} processed tickers to {output_path}")
    print(f"Also added {nasdaq_added_count} tickers from NASDAQ.")
except Exception as e:
    print(f"Error downloading data for multiple tickers: {e}")

Saved 680 processed tickers to ..\data\tickers.csv
Also added 500 tickers from NASDAQ.


#### Calculating CAGR and Scaling ML Data

In [61]:
data_file = path.join("..", "data")
df = pd.read_csv(path.join(data_file, "tickers.csv"))
year_columns = ['2019', '2020', '2021', '2022', '2023', '2024']
cagr_matrix = df[year_columns].pct_change(axis=1).iloc[:, 1:]
cagr_matrix.columns = year_columns[1:]


def extract_growth_features(rates_list):
    rates = np.array(rates_list)
    features = {
        'rate_yr1': rates[0],
        'rate_yr2': rates[1],
        'rate_yr3': rates[2],
        'mean_growth': np.mean(rates),
        'std_growth': np.std(rates),
        'min_growth': np.min(rates),
        'recent_change': rates[2] - rates[1],
        'early_change': rates[1] - rates[0],
        'cumulative_return': np.prod(1 + rates) - 1,
        'negative_years_count': np.sum(rates < 0)
    }
    x = np.array([1, 2, 3])
    slope, _ = np.polyfit(x, rates, 1)
    features['trend_slope'] = slope
    return features


X_list, y_list, split_years = [], [], []
features_A = cagr_matrix[['2020', '2021', '2022']].values
target_A = np.where(cagr_matrix['2023'].values > 0, 1, 0)
features_B = cagr_matrix[['2021', '2022', '2023']].values
target_B = np.where(cagr_matrix['2024'].values > 0, 1, 0)

In [62]:
# Process Window A (Train)
for row, target in zip(features_A, target_A):
    if not np.isnan(row).any():
        X_list.append(extract_growth_features(row))
        y_list.append(target)
        split_years.append('2023')


# Process Window B (Test)
for row, target in zip(features_B, target_B):
    if not np.isnan(row).any():
        X_list.append(extract_growth_features(row))
        y_list.append(target)
        split_years.append('2024')

In [63]:
X_all = pd.DataFrame(X_list)
y_all = np.array(y_list)
split_years = np.array(split_years)
X_train, y_train = X_all[split_years == '2023'], y_all[split_years == '2023']
X_test, y_test = X_all[split_years == '2024'], y_all[split_years == '2024']


#### Model Training

In [66]:
# XGBoost is efficient and dominates short-sequence time-series forecasting
# This model is lightweight and can be trained quickly on a standard CPU
xgc_model = XGBClassifier(
    n_estimators=25, learning_rate=0.04, max_depth=3,
    subsample=0.6, colsample_bytree=0.6, reg_alpha=0.1,
    reg_lambda=1.5, objective='binary:logistic', random_state=102
)
xgc_model.fit(X_train, y_train)
predicts = xgc_model.predict(X_test)
probabilities = xgc_model.predict_proba(X_test)[:, 1]

#### Model Evaluation, With Tolerances

In [67]:
tn, fp, fn, tp = confusion_matrix(y_test, predicts, labels=[0, 1]).ravel()
total_predicted_down = tn + fn
total_predicted_up = fp + tp


print("--- Strict Time-Based Evaluation ---")
print(f"Overall Accuracy: {accuracy_score(y_test, predicts) * 100:.2f}%")
print(f"Precision (When it predicts 'UP'): {precision_score(y_test, predicts) * 100:.2f}%")
print(f"Recall (How many of the actual 'UP' it caught): {recall_score(y_test, predicts) * 100:.2f}%")
print(f"ROC AUC Score: {roc_auc_score(y_test, probabilities):.4f}")
print("------------------------------------")
print(f"Total Predictions: {len(y_test)}")
print(f"Total Predicted 'DOWN': {total_predicted_down} (Correct: {tn}, Incorrect: {fn})")
print(f"Total Predicted 'UP':   {total_predicted_up} (Correct: {tp}, Incorrect: {fp})")

--- Strict Time-Based Evaluation ---
Overall Accuracy: 66.62%
Precision (When it predicts 'UP'): 66.47%
Recall (How many of the actual 'UP' it caught): 99.78%
ROC AUC Score: 0.5237
------------------------------------
Total Predictions: 680
Total Predicted 'DOWN': 6 (Correct: 5, Incorrect: 1)
Total Predicted 'UP':   674 (Correct: 448, Incorrect: 226)
